info txt for running 

In [ ]:
import os
import json
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import timm

warnings.filterwarnings("ignore")


# Configuration
class CFG:
    _BASE_CANDIDATES = [
        "/kaggle/input/competitions/birdclef-2026",
        "/kaggle/input/birdclef-2026"
    ]
    BASE_DIR = next((p for p in _BASE_CANDIDATES if os.path.exists(p)), _BASE_CANDIDATES[0])
    TEST_DIR   = f"{BASE_DIR}/test_soundscapes"
    SAMPLE_SUB = f"{BASE_DIR}/sample_submission.csv"

    DATASET_DIR  = "/kaggle/input/datasets/studentedvard/birdclef-model-v1-zvuk2" 
    WEIGHTS_PATH = f"{DATASET_DIR}/bird_sed_model.pth"
    META_PATH    = f"{DATASET_DIR}/target_columns.json"

    TARGET_SR    = 32_000
    SEGMENT_SEC  = 5.0
    N_MELS       = 128
    N_FFT        = 1024
    HOP_LENGTH   = 320
    FMIN         = 20.0
    FMAX         = 16_000.0

    MODEL_NAME   = "efficientnetv2_s"
    
    DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = CFG()

if not Path(cfg.META_PATH).exists():
    raise FileNotFoundError(f"Metadata file not found: {cfg.META_PATH}")
    
with open(cfg.META_PATH) as f:
    INF_TARGET_COLUMNS = json.load(f)
    
NUM_CLASSES = len(INF_TARGET_COLUMNS)

print(f" Loaded {NUM_CLASSES} bird species classes.")
print(f" Computation device: {cfg.DEVICE}")
# architecture

class BirdSEDModel(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_classes: int,
        pretrained: bool = False,
    ):
        super().__init__()
        
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,         
            in_chans=1,
            num_classes=0,                  
            global_pool="",                 
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, 128, 312)
            feat  = self.backbone(dummy)
            in_ch = feat.shape[1]           

        self.dropout = nn.Dropout(0.3)
        self.fc_clip = nn.Linear(in_ch, num_classes)   
        self.fc_att  = nn.Linear(in_ch, num_classes)   

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)            
        feat = feat.mean(dim=2)            
        feat = feat.permute(0, 2, 1)       
        feat = self.dropout(feat)

        clip_logits = self.fc_clip(feat)                        
        att_weights = torch.softmax(self.fc_att(feat), dim=1)  

        out = (torch.sigmoid(clip_logits) * att_weights).sum(dim=1)
        return out  


def build_mel_transforms(cfg: CFG):
    mel = T.MelSpectrogram(
        sample_rate = cfg.TARGET_SR,
        n_fft       = cfg.N_FFT,
        hop_length  = cfg.HOP_LENGTH,
        n_mels      = cfg.N_MELS,
        f_min       = cfg.FMIN,
        f_max       = cfg.FMAX,
    ).to(cfg.DEVICE)
    db = T.AmplitudeToDB(stype="power", top_db=80).to(cfg.DEVICE)
    return mel, db

def predict_file(path: str, model, mel_t, db_t, cfg: CFG) -> list:
    fname = Path(path).stem
    results = []
    
    try:
        wav, sr = torchaudio.load(path)
        wav = wav.to(cfg.DEVICE)
        
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if sr != cfg.TARGET_SR:
            wav = T.Resample(orig_freq=sr, new_freq=cfg.TARGET_SR).to(cfg.DEVICE)(wav)

        seg_len = int(cfg.SEGMENT_SEC * cfg.TARGET_SR)
        num_frames = wav.shape[1]
        
        n_windows = int(np.ceil(num_frames / seg_len))
        
        for i in range(n_windows):
            start = i * seg_len
            end   = start + seg_len
            segment = wav[:, start:end]

            if segment.shape[1] < seg_len:
                segment = F.pad(segment, (0, seg_len - segment.shape[1]))

            spec = db_t(mel_t(segment))
            spec = (spec - spec.mean()) / (spec.std() + 1e-6)
            spec = spec.unsqueeze(0) 
            with torch.no_grad():
                probs = model(spec).cpu().numpy()[0]
            
            row_id = f"{fname}_{(i + 1) * 5}"
            results.append((row_id, probs))
            
    except Exception as e:
        print(f"erroor {path}: {e}")
        for i in range(24): 
            results.append((f"{fname}_{(i + 1) * 5}", np.zeros(NUM_CLASSES)))

    return results

# Execution
if not Path(cfg.WEIGHTS_PATH).exists():
    raise FileNotFoundError(f"Weights not found: {cfg.WEIGHTS_PATH}")

model = BirdSEDModel(model_name=cfg.MODEL_NAME, num_classes=NUM_CLASSES, pretrained=False)
state_dict = torch.load(cfg.WEIGHTS_PATH, map_location=cfg.DEVICE, weights_only=True)
model.load_state_dict(state_dict, strict=False)
model.to(cfg.DEVICE)
model.eval()
print(f"Model loaded successfully ({cfg.DEVICE}).")

mel_t, db_t = build_mel_transforms(cfg)
test_files  = sorted(glob.glob(f"{cfg.TEST_DIR}/*.ogg"))

sample_sub = pd.read_csv(cfg.SAMPLE_SUB)
SUB_COLS   = list(sample_sub.columns)        
SPECIES_COLS = SUB_COLS[1:]                  

print(f" Test files found: {len(test_files)}")
print(f" Expected bird species columns in submission: {len(SPECIES_COLS)}")
print(f" Model output classes: {NUM_CLASSES}")

model_cols_set = set(INF_TARGET_COLUMNS)
sub_cols_set   = set(SPECIES_COLS)
only_in_model  = model_cols_set - sub_cols_set
only_in_sub    = sub_cols_set - model_cols_set
print(f" In model but not in submission: {len(only_in_model)} → {list(only_in_model)[:5]}...")
print(f" In submission but not in model: {len(only_in_sub)} → {list(only_in_sub)[:5]}...")

if test_files:
    predictions = {}
    for fp in tqdm(test_files, desc="Inference"):
        rows = predict_file(fp, model, mel_t, db_t, cfg)
        for row_id, probs in rows:
            predictions[row_id] = probs
    records = []
    missing_rows = 0
    for row_id in sample_sub["row_id"]:
        record = {"row_id": row_id}
        
        if row_id in predictions:
            probs = predictions[row_id]
            for col in SPECIES_COLS:
                if col in model_cols_set:
                    idx = INF_TARGET_COLUMNS.index(col)
                    record[col] = float(probs[idx])
                else:
                    record[col] = 0.0
        else:
            missing_rows += 1
            for col in SPECIES_COLS:
                record[col] = 0.0
        
        records.append(record)

    if missing_rows > 0:
        print(f" {missing_rows} rows from sample_submission are not covered by predictions → filled with 0.0")

    submission_df = pd.DataFrame(records)[SUB_COLS]
    
    assert list(submission_df.columns) == SUB_COLS, "Columns do not match sample_submission!"
    assert len(submission_df) == len(sample_sub), f"Row count mismatch: {len(submission_df)} vs {len(sample_sub)}"
    
    submission_df.to_csv("submission.csv", index=False)
    print(f" submission.csv saved. Rows: {len(submission_df)}, Columns: {len(submission_df.columns)}")

else:
    print(" Test data is missing. Generating dummy submission.")
    sample_sub.to_csv("submission.csv", index=False)
    print(" Dummy submission.csv saved.")


 Завантажено 234 класів птахів.
 Пристрій обчислень: cpu
Модель завантажена (cpu).
 Тестових файлів: 0
 Очікується 234 колонок птахів у submission.
 Модель видає 234 класів.
 Є в моделі, але не в submission: 0 → []
 Є в submission, але не в моделі: 0 → []
 Тестові дані відсутні. Dummy submission.
 Dummy submission.csv збережено.